In [1]:
import warnings
warnings.filterwarnings('ignore')

In [70]:
import numpy as np
import pandas as pd
import scipy
import matplotlib.pyplot as plt

import re

from tqdm import tqdm

In [3]:
np.__version__, pd.__version__

('2.3.2', '2.3.2')

In [4]:
plt.style.use('ggplot')

In [5]:
# with open("data_1.csv", "r", errors="replace") as f:
#     head = [next(f) for _ in range(5)]
# print(head)

In [6]:
d1 = pd.read_csv('data_1.csv')
d1.head()

,Q3a,Q3b,Q4,Q5,Q6,Q12,Q13,Q14,Q15,Q16,...,Q54,Q54x,Q55,Q56_1,Q56_2,Q56_3,Q56_4,Q56_5,Q56_6,Q56_7
0,3,6,3,3,3,3,4,2,1,2,...,1,3,2,,,,1,,,
1,1,6,1,3,1,2,5,4,4,3,...,2,1,7,1,,,,,,
2,6,,1,1,1,2,4,2,1,2,...,1,3,4,,,,,,,1
3,2,6,2,2,2,4,2,1,3,1,...,2,2,1,1,,,,,,
4,1,5,1,1,1,2,4,2,1,2,...,2,2,4,,,,,,,1


### Exploratory Data Analysis & Wrangling

In [7]:
d1.shape

(8199, 121)

In [8]:
d1 = d1.replace(' ', float('nan'))
d1.head(1)

,Q3a,Q3b,Q4,Q5,Q6,Q12,Q13,Q14,Q15,Q16,...,Q54,Q54x,Q55,Q56_1,Q56_2,Q56_3,Q56_4,Q56_5,Q56_6,Q56_7
0,3,6,3,3,3,3,4,2,1,2,...,1,3,2,NaN,NaN,NaN,1,NaN,NaN,NaN


In [9]:
d1.columns.values

array(['Q3a', 'Q3b', 'Q4', 'Q5', 'Q6', 'Q12', 'Q13', 'Q14', 'Q15', 'Q16',
       'Q17', 'Q20', 'Q21', 'Q22', 'Q23', 'Q24', 'Q25', 'Q26a', 'Q26b_1',
       'Q26b_2', 'Q26b_3', 'Q26b_4', 'Q26b_5', 'AD', 'AGE', 'CD', 'CNTY',
       'CompletionDate', 'F12', 'F13', 'F14', 'F15', 'F16', 'F17', 'F18',
       'F19', 'F20', 'lang', 'latlang', 'likely', 'pid3', 'region',
       'region_coarse', 'RZIP', 'RZIP4', 'SD', 'SEX', 'URBANICITY', 'Q40',
       'Q41', 'Q42', 'Q43_1', 'Q43_2', 'Q43_3', 'Q43_4', 'Q43_5', 'Q43_6',
       'Q43_7', 'Q44', 'Q45', 'Q46', 'Q47_argentina', 'Q47_bolivia',
       'Q47_chile', 'Q47_colombia', 'Q47_costa_rica', 'Q47_cuba',
       'Q47_ecuador', 'Q47_el_salvador', 'Q47_guatemala', 'Q47_honduras',
       'Q47_mexico', 'Q47_nicaragua', 'Q47_others', 'Q47_panama',
       'Q47_paraguay', 'Q47_peru', 'Q47_puerto_rico', 'Q47_uruguay',
       'Q47_venezuela', 'Q48_bangladesh', 'Q48_bhutan', 'Q48_burma',
       'Q48_cambodia', 'Q48_china', 'Q48_guam', 'Q48_hawaii', 'Q48_hmong'

In [10]:
var = 'Q12' # The variable we want to measure using MRP

In [11]:
### FOR Q12
if var == 'Q12':
    d1['Q12'] = d1['Q12'].replace({
        '1':0,
        '2':0,
        '3':1,
        '4':1
    }) # 0 = Approve, 1 = Disapprove

In [12]:
d1[var].value_counts(normalize=True)

Q12
1    0.491397
0    0.486150
5    0.022453
Name: proportion, dtype: float64

In [13]:
poststrat_vars = ['AGE', 'CNTY',
        'pid3', 'SEX', 'Q42', 'Q43_1', 'Q43_2', 'Q43_3', 'Q43_4', 'Q43_5', 'Q43_6',
       'Q43_7', 'Q49', 'Q52', 'Q53', 'Q54x', 'Q55', 'region']

In [14]:
d1['CNTY'].value_counts(normalize=True)

CNTY
19    0.216490
37    0.118673
30    0.076351
1     0.067203
33    0.053665
43    0.045249
7     0.042444
34    0.041834
36    0.040249
38    0.033663
41    0.022198
56    0.020246
49    0.018051
31    0.016465
15    0.015368
10    0.014026
42    0.013416
39    0.013050
48    0.010977
21    0.010123
40    0.009879
27    0.009757
9     0.008050
44    0.008050
50    0.006464
45    0.006098
4     0.005610
57    0.005367
54    0.005245
12    0.004757
13    0.004391
29    0.004025
24    0.003781
28    0.003537
20    0.002683
23    0.002439
16    0.001951
52    0.001951
47    0.001708
55    0.001342
5     0.001342
17    0.001342
51    0.001342
32    0.001220
3     0.001220
35    0.001220
8     0.001098
58    0.000854
11    0.000854
26    0.000488
53    0.000488
6     0.000366
14    0.000366
22    0.000366
25    0.000244
18    0.000244
46    0.000122
Name: proportion, dtype: float64

In [15]:
d1_sub = d1[poststrat_vars + ['Q12']]
d1_sub.head()

,AGE,CNTY,pid3,SEX,Q42,Q43_1,Q43_2,Q43_3,Q43_4,Q43_5,Q43_6,Q43_7,Q49,Q52,Q53,Q54x,Q55,region,Q12
0,21,20,1,2,6,NaN,1,NaN,NaN,NaN,NaN,NaN,1,1,5,3,2,4,1
1,33,7,2,2,7,1,NaN,NaN,NaN,NaN,NaN,NaN,1,1,1,1,7,1,0
2,34,54,1,1,7,1,NaN,NaN,NaN,NaN,NaN,NaN,1,1,5,3,4,4,0
3,22,15,4,2,3,1,NaN,NaN,NaN,NaN,NaN,NaN,1,2,1,2,1,4,1
4,26,30,1,1,8,1,NaN,NaN,NaN,NaN,NaN,NaN,1,NaN,5,2,4,8,0


In [16]:
d1_sub.shape

(8199, 19)

In [17]:
d1_sub['Q12'] = d1_sub['Q12'].fillna('5')
d1_sub = d1_sub[d1_sub['Q12'] != '5']

In [18]:
d1_sub[['Q43_1', 'Q43_2', 'Q43_3', 'Q43_4', 'Q43_5', 'Q43_6',
       'Q43_7']] = d1_sub[['Q43_1', 'Q43_2', 'Q43_3', 'Q43_4', 'Q43_5', 'Q43_6',
       'Q43_7']].fillna(0)
d1_sub.head()

,AGE,CNTY,pid3,SEX,Q42,Q43_1,Q43_2,Q43_3,Q43_4,Q43_5,Q43_6,Q43_7,Q49,Q52,Q53,Q54x,Q55,region,Q12
0,21,20,1,2,6,0,1,0,0,0,0,0,1,1,5,3,2,4,1
1,33,7,2,2,7,1,0,0,0,0,0,0,1,1,1,1,7,1,0
2,34,54,1,1,7,1,0,0,0,0,0,0,1,1,5,3,4,4,0
3,22,15,4,2,3,1,0,0,0,0,0,0,1,2,1,2,1,4,1
4,26,30,1,1,8,1,0,0,0,0,0,0,1,NaN,5,2,4,8,0


In [19]:
race_cols = [k for k in poststrat_vars if k.startswith('Q43')]
race_cols

['Q43_1', 'Q43_2', 'Q43_3', 'Q43_4', 'Q43_5', 'Q43_6', 'Q43_7']

In [20]:
for col in race_cols:
    d1_sub[col] = d1_sub[col].astype(int)

In [21]:
d1_sub['num_race_id'] = np.sum(d1_sub[race_cols], axis=1).astype(int)
d1_sub['num_race_id'].value_counts()

num_race_id
1    7197
2     560
0     182
3      58
4      10
6       2
5       2
Name: count, dtype: int64

In [22]:
d1_sub.iloc[0, :]

AGE            21
CNTY           20
pid3            1
SEX             2
Q42             6
Q43_1           0
Q43_2           1
Q43_3           0
Q43_4           0
Q43_5           0
Q43_6           0
Q43_7           0
Q49             1
Q52             1
Q53             5
Q54x            3
Q55             2
region          4
Q12             1
num_race_id     1
Name: 0, dtype: object

In [23]:
race = np.full(d1_sub.shape[0], fill_value="placeholder")


for i in tqdm(range(d1_sub.shape[0])):
    row = d1_sub.iloc[i, :]
    if row['num_race_id'] > 1:
        race[i] = 'mixed'
    elif row['num_race_id'] == 0:
        race[i] = 'unknown'
    else:
        if row['Q43_1']:
            race[i] = 'white'
        elif row['Q43_2']:
            race[i] = 'hispanic'
        elif row['Q43_3']:
            race[i] = 'aapi'
        elif row['Q43_4']:
            race[i] = 'black'
        elif row['Q43_5'] or row['Q43_6']:
            race[i] = 'natam' # Native American and Alaska Natives
        else:
            race[i] = 'other'

100%|████████████████████████████████████████████████████████████████████████████| 8011/8011 [00:00<00:00, 8395.34it/s]


In [24]:
np.unique(race)

array(['aapi', 'black', 'hispanic', 'mixed', 'natam', 'other', 'unknown',
       'white'], dtype='<U11')

In [25]:
d1_sub['race'] = race

In [26]:
d1_sub['race'].shape

(8011,)

In [27]:
d1_sub['race'].value_counts(normalize=True)

race
white       0.586943
hispanic    0.155037
aapi        0.089377
mixed       0.078892
black       0.054550
unknown     0.022719
other       0.007739
natam       0.004743
Name: proportion, dtype: float64

In [28]:
code_race = {
    'white': 0,
    'black': 1,
    'aapi': 2,
    'hispanic': 3,
    'no_answer': 4,
    'other': 5,
    'natam': 6,
    'mixed': 7
}

d1_sub['race'] = d1_sub['race'].replace(code_race)

In [29]:
d1_sub['race']

0       3
1       0
2       0
3       0
4       0
       ..
8191    3
8193    3
8194    3
8197    3
8198    3
Name: race, Length: 8011, dtype: object

In [30]:
## 1: 18-29, 2: 30-39, 3: 40-49, 4: 50-59, 5: 60-60, 6: 70+
d1_sub['AGE'] = d1_sub['AGE'].map(lambda x: 1 if x < 30 else
                                 (2 if x < 40 else (
                                 3 if x < 50 else (
                                 4 if x < 60 else (
                                 5 if x < 70 else 6)))))

In [31]:
d1_sub['AGE'].value_counts()

AGE
2    1565
1    1411
5    1369
3    1279
6    1227
4    1160
Name: count, dtype: int64

In [32]:
d1_sub['Q52'].value_counts()

Q52
1    4507
2    1775
3     460
Name: count, dtype: int64

In [33]:
d1_sub['Q52'] = d1_sub['Q52'].replace({
    3: 0
}).fillna(0)

In [34]:
d1_sub = d1_sub.drop(race_cols + ['num_race_id'], axis='columns')

In [35]:
d1_sub.head()

,AGE,CNTY,pid3,SEX,Q42,Q49,Q52,Q53,Q54x,Q55,region,Q12,race
0,1,20,1,2,6,1,1,5,3,2,4,1,3
1,2,7,2,2,7,1,1,1,1,7,1,0,0
2,2,54,1,1,7,1,1,5,3,4,4,0,0
3,1,15,4,2,3,1,2,1,2,1,4,1,0
4,1,30,1,1,8,1,0,5,2,4,8,0,0


In [36]:
## Education
## 1: Less than high school grad, 2: High school grad,
## 3: Some college, 4: College grad
d1_sub['Q42'] = d1_sub['Q42'].map(
    lambda x: 1 if x in [1, 2] else (
    2 if x == 3 else (
    3 if x in [4, 5, 6] else 4))
)

In [37]:
d1_sub = d1_sub.drop([
    'pid3', 'Q49', 'Q53', 'Q54x', 'Q55'
], axis='columns')

In [38]:
d1_sub = d1_sub.fillna(0)

In [39]:
d1_sub.head()

,AGE,CNTY,SEX,Q42,Q52,region,Q12,race
0,1,20,2,4,1,4,1,3
1,2,7,2,4,1,1,0,0
2,2,54,1,4,1,4,0,0
3,1,15,2,4,2,4,1,0
4,1,30,1,4,0,8,0,0


In [40]:
demo = pd.read_csv('county_pop_census.csv')
demo.head()

,Label (Grouping),"Alameda County, California","Alpine County, California","Amador County, California","Butte County, California","Calaveras County, California","Colusa County, California","Contra Costa County, California","Del Norte County, California","El Dorado County, California",...,"Sonoma County, California","Stanislaus County, California","Sutter County, California","Tehama County, California","Trinity County, California","Tulare County, California","Tuolumne County, California","Ventura County, California","Yolo County, California","Yuba County, California"
0,Total:,"1,338,388","1,009","34,042","168,339","37,337","15,733","902,672","22,052","152,083",...,"393,152","407,908","74,581","49,851","13,419","331,093","46,268","656,198","171,256","59,545"
1,Population of one race:,"1,208,791",941,"31,277","151,623","33,724","12,898","802,591","19,802","137,417",...,"348,068","351,640","66,593","44,723","12,466","274,662","42,285","555,037","148,054","52,417"
2,White alone,"448,581",736,"26,465","123,122","30,591","7,318","413,652","14,201","120,376",...,"260,998","202,071","37,069","35,744","9,924","139,099","37,273","352,945","89,045","36,082"
3,Black or African American alone,"134,525",9,"1,193","2,773",278,149,"78,292",812,"1,209",...,"6,148","11,770","1,498",323,59,"4,688",983,"12,298","4,813","2,265"
4,American Indian and Alaska Native alone,"14,919",174,578,"3,324",613,351,"9,149","1,618","1,492",...,"6,644","7,772","1,391","1,319",336,"7,431",808,"10,741","2,584","1,289"


In [41]:
demo = demo.T
demo.columns = demo.iloc[0, :]
demo = demo[1:].reset_index()

In [42]:
# pat = re.compile(r'[A-Za-z, ]+\!\!Estimate$')
# index_filt = np.array([d for d in demo['index'].values if pat.match(d)])
# demo = demo[demo['index'].isin(index_filt)]

In [43]:
# demo['index'] = demo['index'].str.rstrip(r'Estimate')
# demo['index'] = demo['index'].str.rstrip(r'!!')

In [44]:
demo.columns = demo.columns.str.lower().str.replace(' ', '_').str.replace(r'[^a-zA-z0-9\(\)]', '').str.replace('\xa0', '')

In [45]:
demo = demo[['index', 'total:']].rename({
    'index': 'county', 'total:': 'vap'
}, axis=1).reset_index()

In [46]:
demo.head()

Label (Grouping),index,county,vap
0,0,"Alameda County, California","1,338,388"
1,1,"Alpine County, California","1,009"
2,2,"Amador County, California","34,042"
3,3,"Butte County, California","168,339"
4,4,"Calaveras County, California","37,337"


In [47]:
demo['vap'] = demo['vap'].str.rstrip(r'\%').str.replace(',', '').astype(float)

In [48]:
demo['county'] = demo['county'].str.rstrip(r', California').map(lambda x: x.upper())

In [49]:
demo.head()

Label (Grouping),index,county,vap
0,0,ALAMEDA COUNTY,1338388.0
1,1,ALPINE COUNTY,1009.0
2,2,AMADOR COUNTY,34042.0
3,3,BUTTE COUNTY,168339.0
4,4,CALAVERAS COUNTY,37337.0


In [50]:
pres_res = pd.read_csv('countypres_2000-2024.csv')
pres_res.head()

,year,state,state_po,county_name,county_fips,office,candidate,party,candidatevotes,totalvotes,version,mode
0,2000,ALABAMA,AL,AUTAUGA,1001.0,US PRESIDENT,AL GORE,DEMOCRAT,4942,17208,20250821,TOTAL
1,2000,ALABAMA,AL,AUTAUGA,1001.0,US PRESIDENT,GEORGE W. BUSH,REPUBLICAN,11993,17208,20250821,TOTAL
2,2000,ALABAMA,AL,AUTAUGA,1001.0,US PRESIDENT,OTHER,OTHER,113,17208,20250821,TOTAL
3,2000,ALABAMA,AL,AUTAUGA,1001.0,US PRESIDENT,RALPH NADER,GREEN,160,17208,20250821,TOTAL
4,2000,ALABAMA,AL,BALDWIN,1003.0,US PRESIDENT,AL GORE,DEMOCRAT,13997,56480,20250821,TOTAL


In [51]:
pres_res = pres_res[
    (pres_res['year'] == 2020) & (pres_res['state_po'] == 'CA') & (pres_res['party'] == 'DEMOCRAT')
]

In [52]:
pres_res['county_full'] = pres_res['county_name'] + ' COUNTY'

In [53]:
pres_res.head()

,year,state,state_po,county_name,county_fips,office,candidate,party,candidatevotes,totalvotes,version,mode,county_full
52657,2020,CALIFORNIA,CA,ALAMEDA,6001.0,US PRESIDENT,JOSEPH R BIDEN JR,DEMOCRAT,617659,770070,20250821,TOTAL,ALAMEDA COUNTY
52662,2020,CALIFORNIA,CA,ALPINE,6003.0,US PRESIDENT,JOSEPH R BIDEN JR,DEMOCRAT,476,741,20250821,TOTAL,ALPINE COUNTY
52667,2020,CALIFORNIA,CA,AMADOR,6005.0,US PRESIDENT,JOSEPH R BIDEN JR,DEMOCRAT,8153,22304,20250821,TOTAL,AMADOR COUNTY
52672,2020,CALIFORNIA,CA,BUTTE,6007.0,US PRESIDENT,JOSEPH R BIDEN JR,DEMOCRAT,50426,102066,20250821,TOTAL,BUTTE COUNTY
52677,2020,CALIFORNIA,CA,CALAVERAS,6009.0,US PRESIDENT,JOSEPH R BIDEN JR,DEMOCRAT,10046,27164,20250821,TOTAL,CALAVERAS COUNTY


In [54]:
pres_res = pd.merge(left=pres_res, right=demo, left_on='county_full', right_on='county', how='left')
pres_res.head()

,year,state,state_po,county_name,county_fips,office,candidate,party,candidatevotes,totalvotes,version,mode,county_full,index,county,vap
0,2020,CALIFORNIA,CA,ALAMEDA,6001.0,US PRESIDENT,JOSEPH R BIDEN JR,DEMOCRAT,617659,770070,20250821,TOTAL,ALAMEDA COUNTY,0,ALAMEDA COUNTY,1338388.0
1,2020,CALIFORNIA,CA,ALPINE,6003.0,US PRESIDENT,JOSEPH R BIDEN JR,DEMOCRAT,476,741,20250821,TOTAL,ALPINE COUNTY,1,ALPINE COUNTY,1009.0
2,2020,CALIFORNIA,CA,AMADOR,6005.0,US PRESIDENT,JOSEPH R BIDEN JR,DEMOCRAT,8153,22304,20250821,TOTAL,AMADOR COUNTY,2,AMADOR COUNTY,34042.0
3,2020,CALIFORNIA,CA,BUTTE,6007.0,US PRESIDENT,JOSEPH R BIDEN JR,DEMOCRAT,50426,102066,20250821,TOTAL,BUTTE COUNTY,3,BUTTE COUNTY,168339.0
4,2020,CALIFORNIA,CA,CALAVERAS,6009.0,US PRESIDENT,JOSEPH R BIDEN JR,DEMOCRAT,10046,27164,20250821,TOTAL,CALAVERAS COUNTY,4,CALAVERAS COUNTY,37337.0


In [55]:
pres_res['biden_vote_pct'] = pres_res['candidatevotes'] / pres_res['vap']
# pres_res = pres_res.reset_index().reset_index()
pres_res.head()

,year,state,state_po,county_name,county_fips,office,candidate,party,candidatevotes,totalvotes,version,mode,county_full,index,county,vap,biden_vote_pct
0,2020,CALIFORNIA,CA,ALAMEDA,6001.0,US PRESIDENT,JOSEPH R BIDEN JR,DEMOCRAT,617659,770070,20250821,TOTAL,ALAMEDA COUNTY,0,ALAMEDA COUNTY,1338388.0,0.461495
1,2020,CALIFORNIA,CA,ALPINE,6003.0,US PRESIDENT,JOSEPH R BIDEN JR,DEMOCRAT,476,741,20250821,TOTAL,ALPINE COUNTY,1,ALPINE COUNTY,1009.0,0.471754
2,2020,CALIFORNIA,CA,AMADOR,6005.0,US PRESIDENT,JOSEPH R BIDEN JR,DEMOCRAT,8153,22304,20250821,TOTAL,AMADOR COUNTY,2,AMADOR COUNTY,34042.0,0.239498
3,2020,CALIFORNIA,CA,BUTTE,6007.0,US PRESIDENT,JOSEPH R BIDEN JR,DEMOCRAT,50426,102066,20250821,TOTAL,BUTTE COUNTY,3,BUTTE COUNTY,168339.0,0.299550
4,2020,CALIFORNIA,CA,CALAVERAS,6009.0,US PRESIDENT,JOSEPH R BIDEN JR,DEMOCRAT,10046,27164,20250821,TOTAL,CALAVERAS COUNTY,4,CALAVERAS COUNTY,37337.0,0.269063


In [56]:
pres_res = pres_res.rename({
    'index': 'level_0'
}, axis='columns')

In [57]:
pres_res['level_0'] = pres_res['level_0'] + 1

In [58]:
pres_res['countyicp'] = '71' + pres_res['county_fips'].astype(int).astype(str).str.extract(r'\d(\d+)')

In [59]:
pres_res.head()

,year,state,state_po,county_name,county_fips,office,candidate,party,candidatevotes,totalvotes,version,mode,county_full,level_0,county,vap,biden_vote_pct,countyicp
0,2020,CALIFORNIA,CA,ALAMEDA,6001.0,US PRESIDENT,JOSEPH R BIDEN JR,DEMOCRAT,617659,770070,20250821,TOTAL,ALAMEDA COUNTY,1,ALAMEDA COUNTY,1338388.0,0.461495,71001
1,2020,CALIFORNIA,CA,ALPINE,6003.0,US PRESIDENT,JOSEPH R BIDEN JR,DEMOCRAT,476,741,20250821,TOTAL,ALPINE COUNTY,2,ALPINE COUNTY,1009.0,0.471754,71003
2,2020,CALIFORNIA,CA,AMADOR,6005.0,US PRESIDENT,JOSEPH R BIDEN JR,DEMOCRAT,8153,22304,20250821,TOTAL,AMADOR COUNTY,3,AMADOR COUNTY,34042.0,0.239498,71005
3,2020,CALIFORNIA,CA,BUTTE,6007.0,US PRESIDENT,JOSEPH R BIDEN JR,DEMOCRAT,50426,102066,20250821,TOTAL,BUTTE COUNTY,4,BUTTE COUNTY,168339.0,0.299550,71007
4,2020,CALIFORNIA,CA,CALAVERAS,6009.0,US PRESIDENT,JOSEPH R BIDEN JR,DEMOCRAT,10046,27164,20250821,TOTAL,CALAVERAS COUNTY,5,CALAVERAS COUNTY,37337.0,0.269063,71009


In [60]:
pres_res.to_csv('ca_pres_res.csv')

In [61]:
d1_sub = pd.merge(left=d1_sub, right=pres_res[['level_0', 'biden_vote_pct']], left_on='CNTY', right_on='level_0', how='left')
d1_sub.head()

,AGE,CNTY,SEX,Q42,Q52,region,Q12,race,level_0,biden_vote_pct
0,1,20,2,4,1,4,1,3,20,0.204612
1,2,7,2,4,1,1,0,0,7,0.461282
2,2,54,1,4,1,4,0,0,54,0.199657
3,1,15,2,4,2,4,1,0,15,0.204631
4,1,30,1,4,0,8,0,0,30,0.323063


In [62]:
d1_sub = d1_sub.drop(['level_0'], axis='columns')

In [63]:
d1_sub = d1_sub.rename({
    'Q42': 'edu',
    'Q52': '2020_vote'
}, axis=1)

In [64]:
d1_sub.shape

(8011, 9)

In [65]:
d1_sub['Q12']

0       1
1       0
2       0
3       1
4       0
       ..
8006    0
8007    1
8008    1
8009    0
8010    1
Name: Q12, Length: 8011, dtype: int64

In [66]:
d1_sub.head()

,AGE,CNTY,SEX,edu,2020_vote,region,Q12,race,biden_vote_pct
0,1,20,2,4,1,4,1,3,0.204612
1,2,7,2,4,1,1,0,0,0.461282
2,2,54,1,4,1,4,0,0,0.199657
3,1,15,2,4,2,4,1,0,0.204631
4,1,30,1,4,0,8,0,0,0.323063


In [67]:
np.unique(d1_sub['Q12'])

array([0, 1])

In [68]:
d1_sub.shape

(8011, 9)

In [81]:
d1['region'].value_counts()

region
1    2078
6    1775
9     973
5     806
3     728
8     626
4     513
2     513
7     187
Name: count, dtype: int64

In [82]:
region_to_cnty = pd.DataFrame(d1[['region', 'CNTY']].value_counts().fillna(0))
region_to_cnty = region_to_cnty.reset_index()
region_to_cnty

,region,CNTY,count
0,6,19,1775
1,9,37,973
2,8,30,626
3,1,1,551
4,5,33,440
5,1,43,371
6,1,7,348
7,3,34,343
8,5,36,330
9,1,38,276


In [83]:
region_to_cnty['CNTY'].value_counts()

CNTY
19    1
37    1
30    1
1     1
33    1
43    1
7     1
34    1
36    1
38    1
41    1
56    1
49    1
31    1
15    1
10    1
42    1
39    1
48    1
21    1
40    1
27    1
44    1
9     1
50    1
45    1
4     1
57    1
54    1
12    1
13    1
29    1
24    1
28    1
20    1
23    1
16    1
52    1
47    1
17    1
51    1
55    1
5     1
35    1
32    1
3     1
8     1
58    1
11    1
53    1
26    1
6     1
22    1
14    1
18    1
25    1
46    1
Name: count, dtype: int64

In [84]:
region_to_cnty.to_csv('region_to_cnty.csv')

In [88]:
cnty_num_key = pres_res[['county_name', 'level_0']]
cnty_num_key = cnty_num_key.rename({'level_0': 'cnty'}, axis=1)
cnty_num_key.head()

,county_name,cnty
0,ALAMEDA,1
1,ALPINE,2
2,AMADOR,3
3,BUTTE,4
4,CALAVERAS,5


In [89]:
cnty_num_key.to_csv('cnty_num_key.csv')

In [69]:
## Change name of CSV if measuring for other questions!
d1_sub.to_csv('reg_dfs/reg_df_Q12.csv')